In [ ]:
import pandas as pd
from astroquery.gaia import Gaia
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
data_dir = '/Users/z5214708/OneDrive - UNSW/'

In [ ]:
#read in true sources
cs = pd.read_csv('../data/true_sources_final.csv')

In [ ]:
#calc g mag error based on flux error 
cs['phot_g_mean_mag_error'] = (1.0857 * (cs['phot_g_mean_flux_error']/cs['phot_g_mean_flux'])).clip(lower=0.1) 

In [ ]:
#get 2mass photometry
cache_path = Path('../data/tmass_query_results.csv')
if cache_path.exists():
    tmass = pd.read_csv(cache_path)
else:
    ids = (
        cs["source_id"]
        .dropna()
        .astype("int64")
        .astype(str)
        .drop_duplicates()
        .tolist()
    )

    batch_size = 500
    tmass_results = []

    for batch_number, start in enumerate(range(0, len(ids), batch_size), start=1):

        batch = ids[start:start + batch_size]
        id_string = ",".join(batch)

        query = f"""
            SELECT
                xmatch.source_id AS source_id,
                tmass.designation AS tmass_designation,

                tmass.j_m,
                tmass.j_msigcom AS j_err,

                tmass.h_m,
                tmass.h_msigcom AS h_err,

                tmass.ks_m,
                tmass.ks_msigcom AS ks_err,

                tmass.ph_qual

            FROM gaiadr3.tmass_psc_xsc_best_neighbour AS xmatch

            JOIN gaiadr3.tmass_psc_xsc_join AS xjoin
                ON xmatch.clean_tmass_psc_xsc_oid =
                xjoin.clean_tmass_psc_xsc_oid

            JOIN gaiadr1.tmass_original_valid AS tmass
                ON xjoin.original_psc_source_id =
                tmass.designation

            WHERE xmatch.source_id IN ({id_string})
            """

        print(f"Running batch {batch_number}...")

        try:
            job = Gaia.launch_job_async(query)
            result = job.get_results().to_pandas()
            tmass_results.append(result)

        except Exception as error:
            print(f"Batch {batch_number} failed:")
            print(error)
            print(query)
            raise

    tmass = pd.concat(tmass_results, ignore_index=True)
    tmass.to_csv(cache_path,index=False)
cs  = cs.merge(tmass,on='source_id',how='left')

In [ ]:
#get evolutionary state classifcations
rgb_rc = pd.read_csv(data_dir+'rgb_rc_cleaned_classification.csv')

In [ ]:
cs = cs.merge(rgb_rc[['ID','ev_state']],left_on='fname',right_on='ID',how='left')
cs['nike_y'] = cs['numax']**0.75/cs['dnu']
cs['ev_state'] = cs['ev_state'].map({1: 'clump', 0: 'rgb'})
cs.loc[(cs['ev_state'] == 'rgb') & (cs['nike_y'] < 3.2), 'ev_state'] = float('nan')
cs.loc[(cs['ev_state'] == 'clump') & (cs['numax'] < 15), 'ev_state'] = 'agb'

In [ ]:
#clean columns
cs = cs[['source_id', 'numax', 'numax_sig', 'dnu', 'dnu_sig', 'Teff', 's_Teff', 'feh', 's_feh', 
       'ra', 'dec', 
       'phot_g_mean_mag', 'phot_g_mean_mag_error', 'j_m', 'j_err', 'h_m', 'h_err', 'ks_m', 'ks_err', 'ev_state','ph_qual','syd_success']]

In [ ]:
#masks to select different subsets

#do we have the phase
phase_known = cs["ev_state"].notna()

#is 2mass quality AAA (if so use)
use_tmass = (
    cs["ph_qual"]
    .fillna("")
    .str.strip()
    .eq("AAA")
)

#is the dnu measurement good (syd success == 2)
good_dnu = (
    pd.to_numeric(cs["syd_success"], errors="coerce").eq(2)
    & cs["dnu"].notna()
    & cs["dnu_sig"].notna()
)

In [ ]:
cs = cs.iloc[:,:-2]

In [ ]:
cases = {
    "a": use_tmass & phase_known,
    "b": use_tmass & ~phase_known,
    "c": ~use_tmass & phase_known,
    "d": ~use_tmass & ~phase_known,
}

In [ ]:
#four tables
#no numax correction
one = cs.copy()
two = cs[good_dnu].copy()

#numax correction
correction_factor = factor = -0.0588 * cs["feh"] + 0.949
corrected_cs = cs.copy()
corrected_cs['numax'] = factor*cs['numax']
corrected_cs['numax_sig'] = np.sqrt((factor*cs['numax_sig'])**2 + (0.0588*cs['numax']*cs['s_feh'])**2)

three = corrected_cs.copy()
four = corrected_cs[good_dnu].copy()


In [ ]:
save_dir = '/Users/z5214708/BASTA/BASTA/final_runs/data/'

In [ ]:
for case, mask in cases.items():
    one[mask].to_csv(save_dir+f'raw_{case}.ascii',index=False,sep=" ", header=False,na_rep="-999.999")
    two[mask].to_csv(save_dir+f'raw_dnu_{case}.ascii',index=False,sep=" ", header=False,na_rep="-999.999")
    three[mask].to_csv(save_dir+f'corrected_{case}.ascii',index=False,sep=" ", header=False,na_rep="-999.999")
    four[mask].to_csv(save_dir+f'corrected_dnu_{case}.ascii',index=False,sep=" ", header=False,na_rep="-999.999")
